# Session 2 Workshop: Learning a Shape Space With No Data

**Paper:** Sharp et al. (2023), *Data-Free Learning of Reduced-Order Kinematics* (SIGGRAPH 2023)

This notebook runs alongside the slides. The talk stops 14 times for a **code break**: each one is a numbered task below. Each code break comes right after the slide that answers a question, so the numbers your code prints should match the ones you just saw on screen.

| Code break | Task | Goes with slides |
|---|---|---|
| 1 | Build the bar and its energy | Part 1: "Energy Tells Good Shapes from Bad" |
| 2 | Linear modes from the Hessian | Part 2: "What Is a Mode?" |
| 3 | Minimise energy alone | Part 3: "Objective 1: Low Expected Energy" |
| 4 | Distance keeping and the seed | Part 3: "Seeded Exploration" |
| 5 | What σ does | Part 3: "Choosing σ and λ" |
| 6 | Linear vs. neural, far from rest | Part 3: "Linear vs. Neural on Our Bar" |
| 7 | Simulate inside the subspace | Part 4: "Simulating Inside the Subspace" |
| 8 | One network for every pin distance | Part 4: "Conditional Subspaces" |
| 9 | A floor, as a penalty | Part 4: "Constraints and Collisions as Penalties" |
| 10 | Rigid bodies as penalties | Part 4: "Rigid Bodies Without Special Code" |
| 11 | Rotation: linear vs. neural | Part 5: "Neural Subspaces Can Rotate" |
| 12 | Sampled shapes as training data | Part 5: "Data-Free vs. a Trained Autoencoder" |
| 13 | Two latent numbers | Part 5: "Going Deeper" |
| 14 | Seeds and restarts | Part 5: "Limitations" |

No dataset is used anywhere. The only inputs are an energy function and one seed shape.

**How to use it:** File → Save a copy in Drive, then run the cells from the top. A CPU runtime is enough; every training run takes seconds. Cells marked **Your job** have a `# YOUR CODE HERE` line to fill in.

In [ ]:
import math, time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
cm = plt.cm.viridis
print("PyTorch", torch.__version__)
def check(ok, what):
    print(("PASS  " if ok else "FAIL  ") + what)

## Before you start: geometry and PyTorch cheat sheet

You do **not** need a geometry background. Everything in this notebook uses the handful of ideas below. Each task also has a short primer, three hints that get more specific (click to open), and a **check** cell that tells you whether your answer is right.

**Vectors and shapes**
- A 2D point or arrow is two numbers (x, y). In code the last axis holds them: `p[..., 0]` is x, `p[..., 1]` is y.
- An *edge vector* is "end point minus start point": `e = X[k+1] - X[k]`. Its length is `e.norm(dim=-1)`.
- A whole shape is 36 numbers in one row; `full_nodes(q)` turns it into 20 points of shape `(batch, 20, 2)`.

**Products of two vectors a and b (2D)**

| name | formula | means |
|---|---|---|
| dot | aₓbₓ + a_y b_y | \|a\|\|b\| cos θ: big when they point the same way |
| cross | aₓb_y − a_y bₓ | \|a\|\|b\| sin θ: sign says left (+) or right (−) turn |
| angle | atan2(cross, dot) | θ between them, from −180° to +180° |

**Distance between two shapes** is just the length of their difference: `(q1 - q2).norm()`, treating the 36 numbers as one long vector.

**Shapes of tensors (the most common bug)**
- `x[:, None]` adds an axis: `(32,)` becomes `(32, 1)`. Needed before joining or multiplying with a matrix.
- `torch.cat([a, b], dim=1)` puts columns side by side: `(32, 1)` and `(32, 1)` give `(32, 2)`.
- Broadcasting: `(51, 1) * (36,)` gives `(51, 36)`: every row gets its own scale.
- `A @ B` is matrix multiplication; `R.transpose(-1, -2)` swaps the last two axes (the matrix transpose).
- `.sum(dim)`, `.mean(dim)`: add or average along one axis. `.sum()` with no axis gives one number.

**If something fails:** print `.shape` of every tensor in the line. Almost every error here is a shape mismatch.

## Task 1 · Build the bar and its energy  *(code break 1)*

> **Slides:** Part 1, "A Shape Is One Point in ℝⁿ" and "Energy Tells Good Shapes from Bad". The question slide before this break asks what happens to the energy when every node is nudged a little.

Our bar is a chain of **N = 20 nodes** in 2D. The end nodes are pinned at (0, 0) and (0.7, 0). The bar's natural length is 1, so it is squeezed to 70% and has to bend.

Only the 18 middle nodes move, each with an (x, y) position, so one shape is a vector **q with n = 36 numbers**: a point in ℝ³⁶.

The energy has three parts:

- **Stretching:** each edge wants its rest length l₀. Cost kₛ (|e| − l₀)² / l₀ per edge.
- **Bending:** each joint wants to be straight. Cost k_b θ² / l₀, where θ is the turning angle between neighbouring edges.
- **Gravity:** a small pull down, g × (average height).

**Your job:** write `turning_angle(a, b)`: the signed angle between two edge vectors. The `energy` function below then calls it at every joint of every shape. Read the geometry primer and the hints first if angles from vectors are new to you.

### Reading the equation

> **E(q) = Σ stretch + Σ bend + gravity + penalties**

*Read it as:* one number that scores a shape: small for shapes the bar would really take, huge for nonsense.

- **Σ** → add up over every edge or joint
- **stretch** → edge longer or shorter than its rest length
- **bend** → joint not straight
- **penalties** → extra rules, e.g. a floor (Part 4)

### Geometry primer: the angle at a joint

At every joint, the bar arrives along edge **a** and leaves along edge **b**. If the bar is straight, b points the same way as a and the angle is 0. If it bends left, the angle is positive; right, negative.

Two numbers tell you everything:

- **dot** = aₓbₓ + a_y b_y = |a||b| cos θ
- **cross** = aₓb_y − a_y bₓ = |a||b| sin θ

`atan2(sin-like, cos-like)` turns them back into θ, with the correct sign, and the lengths |a||b| cancel out. Worked example: a = (1, 0), b = (0, 1). dot = 0, cross = 1, so θ = atan2(1, 0) = 90°: a left turn.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

You need one number per joint: how far the direction turns from edge a to edge b. Compute the 2D cross product and the dot product, then `torch.atan2(cross, dot)`.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`a` and `b` have shape `(..., 2)`. `a[..., 0]` is every x component, `a[..., 1]` every y. The result should have the shape of `a` without its last axis. `(a * b).sum(-1)` multiplies matching components and adds them.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
cross = a[..., 0] * b[..., 1] - a[..., 1] * b[..., 0]
dot = (a * b).sum(-1)
return torch.atan2(cross, dot)
```

</details>

In [ ]:
def turning_angle(a, b):
    # a, b: edge vectors with shape (..., 2), last axis = (x, y).
    # returns the signed angle (radians) that turns a onto b: 0 if straight, + for a left turn, - for a right turn
    # YOUR CODE HERE
    raise NotImplementedError("Fill in this step")

**Check your answer:** run this cell. Every line should say PASS.

In [ ]:
deg = lambda x: round(math.degrees(float(x)))
a = torch.tensor([1.0, 0.0])
check(deg(turning_angle(a, torch.tensor([1.0, 0.0]))) == 0,    "straight joint gives 0 degrees")
check(deg(turning_angle(a, torch.tensor([0.0, 1.0]))) == 90,   "left turn gives +90 degrees")
check(deg(turning_angle(a, torch.tensor([1.0, -1.0]))) == -45, "right turn gives -45 degrees")
check(turning_angle(torch.ones(4, 7, 2), torch.ones(4, 7, 2)).shape == (4, 7), "works on a whole batch of joints")

In [ ]:
N, L = 20, 1.0                 # nodes, natural length of the bar
l0 = L / (N - 1)               # rest length of one edge
COMP = 0.7                     # right pin sits at 0.7 * L: the bar is squeezed
k_s, k_b, g = 100.0, 0.05, 0.1 # stretch stiffness, bend stiffness, gravity
n = 2 * (N - 2)                # free numbers per shape: 18 nodes x (x, y)

def full_nodes(q, c=COMP):
    # q: (batch, n) free coordinates  ->  (batch, N, 2) all node positions, pins included
    B = q.shape[0]
    if not torch.is_tensor(c):
        c = torch.full((B,), c)
    left = torch.zeros(B, 1, 2)
    right = torch.stack([c * L, torch.zeros_like(c)], -1)[:, None]
    return torch.cat([left, q.view(B, N - 2, 2), right], 1)

def energy(q, c=COMP):
    X = full_nodes(q, c)
    e = X[:, 1:] - X[:, :-1]                        # edge vectors, (batch, N-1, 2)
    E_stretch = k_s * ((e.norm(dim=-1) - l0) ** 2).sum(1) / l0

    a, b = e[:, :-1], e[:, 1:]                      # consecutive edge pairs, each (batch, N-2, 2)
    theta = turning_angle(a, b)                     # bend angle at every joint, (batch, N-2)
    E_bend = k_b * (theta ** 2).sum(1) / l0

    E_grav = g * X[..., 1].mean(1)
    return E_stretch + E_bend + E_grav

def seed_shape(c=COMP):
    xs = torch.linspace(0, c * L, N)[1:-1]
    return torch.stack([xs, torch.zeros_like(xs)], 1).reshape(-1)

q_seed = seed_shape()          # the straight, squeezed bar
print("numbers per shape n =", n)
print(f"energy of the straight seed shape : {energy(q_seed[None]).item():8.3f}")
print(f"energy of a random nearby shape   : {energy(q_seed[None] + 0.05 * torch.randn(1, n)).item():8.3f}")

Now find the two shapes the bar actually wants to be in. We start just above and just below the straight seed and let plain gradient descent settle each one. This is **not** part of the method; it only gives reference shapes for later.

In [ ]:
def settle(q0, steps=3000, lr=1e-2):
    q = q0.clone().requires_grad_(True)
    opt = torch.optim.Adam([q], lr)
    for _ in range(steps):
        opt.zero_grad(); energy(q[None]).sum().backward(); opt.step()
    return q.detach()

xs = torch.linspace(0, COMP * L, N)[1:-1]
bump = torch.stack([torch.zeros_like(xs), torch.sin(math.pi * xs / (COMP * L))], 1).reshape(-1)
q_up, q_down = settle(q_seed + 0.1 * bump), settle(q_seed - 0.1 * bump)

def draw(ax, q, c=COMP, **kw):
    X = full_nodes(q[None], c)[0]
    ax.plot(X[:, 0], X[:, 1], "-o", ms=3, **kw)
    ax.set_aspect("equal"); ax.set_xlim(-0.05, 0.95); ax.set_ylim(-0.45, 0.45)

fig, axs = plt.subplots(1, 4, figsize=(16, 3))
for ax, q, t in zip(axs, [q_seed, q_seed + 0.05 * torch.randn(n), q_up, q_down],
                    ["seed (straight)", "random nearby", "settled: buckled up", "settled: buckled down"]):
    draw(ax, q); ax.set_title(f"{t}\nE = {energy(q[None]).item():.2f}")
plt.show()
print(f"buckled-up energy {energy(q_up[None]).item():.3f}, buckled-down energy {energy(q_down[None]).item():.3f}")

**Question:** a random shape *near* the seed has an energy dozens of times higher than the seed. Why? What does that say about where the useful shapes are in ℝ³⁶?

## Task 2 · Linear modes from the Hessian  *(code break 2)*

> **Slides:** Part 2, "Linear Modal Analysis" and "What Is a Mode?"

Classic modal analysis builds a *linear* subspace around one rest shape b:

f(z) = b + σ A z,

where the columns of A are the eigenvectors of the energy's Hessian H(b) with the smallest eigenvalues: the cheapest directions to move in.

**Your job:** compute the Hessian of the energy at `q_down` with `torch.autograd.functional.hessian`, get its eigenvalues and eigenvectors with `torch.linalg.eigh` (they come sorted, smallest first), and keep the softest eigenvector as `A` (shape n × 1).

### Reading the equation

> **H v = λ v**

*Read it as:* push the bar a little in direction v; it resists along that same direction, with stiffness λ.

- **H** → Hessian: curvature of the energy in every direction
- **v** → a direction (eigenvector)
- **λ** → how steep the energy bowl is that way
- **mode 1** → smallest λ, the cheapest move (3.0 vs 27.5)

> **f(z) = A z + x₀**

*Read it as:* start at a rest shape and move in a straight line along fixed directions.

- **x₀** → the rest shape (here buckled down)
- **A** → the softest directions: Hessian eigenvectors
- **z** → how far to move along each one
- **small |z|** → a good local model
- **large |z|** → edges stretch: shears and scales, cannot rotate

### Geometry primer: the Hessian and its eigenvectors, without the jargon

Stand at the buckled-down shape and try moving in some direction v (a list of 36 small nudges). For small steps s the energy rises like a bowl: E ≈ E₀ + ½ λ s². Different directions give bowls of different steepness λ.

- The **Hessian** H is a 36 × 36 table of second derivatives that stores the steepness of every direction at once.
- An **eigenvector** v of H is a direction where the bowl is "pure": moving along v keeps you pointing along v. Its **eigenvalue** λ is that bowl's steepness.
- The eigenvector with the **smallest** λ is the cheapest way to move: the first mode.

You never compute any of this by hand; PyTorch does both steps.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

Two library calls: one builds H from the energy function, the other splits H into eigenvalues and eigenvectors. Then keep the first eigenvector.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`torch.autograd.functional.hessian(f, x)` needs `f` to take **one** shape of shape `(36,)` and return **one** number. Our `energy` takes a batch, so wrap it: `lambda q: energy(q[None]).sum()`. `torch.linalg.eigh(H)` returns `(values, vectors)` with values sorted smallest first; eigenvector k is the **column** `vectors[:, k]`. Keep `A` two-dimensional, shape `(36, 1)`, by slicing `[:, :1]`.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
H = torch.autograd.functional.hessian(lambda q: energy(q[None]).sum(), q_down)
evals, evecs = torch.linalg.eigh(H)
A = evecs[:, :1]
```

</details>

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Fill in this step")
print("four smallest Hessian eigenvalues:", [round(v, 2) for v in evals[:4].tolist()])

def linear_map(z, sigma=3.0):
    return q_down + sigma * z @ A.T

fig, axs = plt.subplots(1, 2, figsize=(14, 2.8))
with torch.no_grad():
    for ax, zmax in zip(axs, [0.1, 0.6]):
        zz = torch.linspace(-zmax, zmax, 5)[:, None]
        Q = linear_map(zz)
        for k, q in enumerate(Q): draw(ax, q, color=cm(k / 4))
        ax.set_title(f"linear modal subspace, |z| <= {zmax}:  max energy {energy(Q).max().item():.1f}")
plt.show()

**Check your answer:** run this cell. Every line should say PASS.

In [ ]:
check(H.shape == (n, n), "H is 36 x 36")
check(torch.allclose(H, H.T, atol=1e-4), "H is symmetric, as every Hessian is")
check(abs(evals[0].item() - 2.98) < 0.05, f"softest eigenvalue is about 2.98 (yours: {evals[0].item():.2f})")
check(A.shape == (n, 1) and abs(A.norm().item() - 1) < 1e-4, "A is one unit-length column")

**Question:** with |z| ≤ 0.1 the shapes are fine. What goes wrong at |z| = 0.6, only six times further along the same direction?

## Task 3 · Minimise energy alone  *(code break 3)*

> **Slides:** Part 3, "A Neural Subspace Map" and "Objective 1: Low Expected Energy". The question slide before this break asks how different the 7 output shapes will be.

A **subspace map** is a small network f_θ that turns a latent z (here d = 1) into a full shape q = f_θ(z) in ℝⁿ. The paper's first idea (Eq. 1) makes those shapes low-energy on average:

min over θ of 𝔼_z [ E(f_θ(z)) ],  z ∼ N(0, I).

**Your job:** write the loss inside `train_energy_only`: the mean energy of the batch `q`.

### Reading the equation

> **min over θ    𝔼 z∼N(0, I) [ E( f_θ(z) ) ]**

*Read it as:* tune the network so the shapes it makes from random inputs have low energy on average. No data anywhere.

- **θ** → the network's weights
- **z ∼ N(0, I)** → random latent numbers, mostly near 0
- **f_θ(z)** → the shape made from z
- **𝔼[ ]** → average over many z

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

Eq. 1 says: make the average energy of the generated shapes small. So the loss is the average of `energy(q)` over the batch.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`energy(q)` returns one number per shape, shape `(32,)`. `.mean()` turns that into the single number that `loss.backward()` needs.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
loss = energy(q).mean()
```

</details>

In [ ]:
class SubspaceNet(nn.Module):
    # MLP from latent z (and an optional condition) to the free coordinates; ELU, as in the paper
    def __init__(self, d, cond_dim=0, hidden=64, out=n):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d + cond_dim, hidden), nn.ELU(),
            nn.Linear(hidden, hidden), nn.ELU(),
            nn.Linear(hidden, hidden), nn.ELU(),
            nn.Linear(hidden, out),
        )
    def forward(self, z):
        return self.net(z)

def train_energy_only(d=1, steps=2000, batch=32):
    net = SubspaceNet(d)
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    for it in range(steps):
        z = torch.randn(batch, d)
        q = net(z)
        # YOUR CODE HERE
        raise NotImplementedError("Fill in this step")
        opt.zero_grad(); loss.backward(); opt.step()
    return net

torch.manual_seed(0)
net_naive = train_energy_only()
with torch.no_grad():
    Q = net_naive(torch.linspace(-3, 3, 7)[:, None])
print("energies along z  :", [round(e, 3) for e in energy(Q).tolist()])
print(f"spread of shapes  : {Q.std(0).norm().item():.5f}   (0 means every z gives the same shape)")

fig, ax = plt.subplots(figsize=(5, 2.5))
for q in Q: draw(ax, q, alpha=0.6)
ax.set_title("energy-only subspace: 7 values of z"); plt.show()

**Question:** every z gives (almost) the same shape. Why is that the "best" answer to Eq. 1, and why is it useless?

## Task 4 · Distance keeping and the seed  *(code break 4)*

> **Slides:** Part 3, "Keep Distances", "Distance Keeping in Plain Words", "The Full Objective" and "Seeded Exploration". The question slide before this break asks whether one number can reach both buckled states.

The paper adds a term that asks the map to keep distances up to a scale σ (Eq. 3). For two latent samples z and z′:

( log( |f(z) − f(z′)| / (σ |z − z′|) ) )²

It is zero when the shapes are exactly σ times as far apart as the latent points. We average it over every pair in the batch and weight it by λ (Eq. 4). Our mass matrix is the identity, so |·|_M is the ordinary length.

The **seeded schedule** (Eq. 8) trains

f(z) = ρ · MLP(z) + (1 − ρ) · q_seed,

with ρ rising from 0 to 1 and target scale ρσ. At the start every shape is the seed, so energies are sane; the subspace grows outward.

**Your jobs:** write `isometry_loss` and `subspace_map`.

### Reading the equation

> **Eq. 2:   |f(z) − f(z′)|_M ≈ σ |z − z′|**

*Read it as:* inputs that are far apart must give shapes that are far apart, so the map cannot collapse to one shape.

- **|z − z′|** → how far apart two inputs are
- **|f(z) − f(z′)|_M** → how far apart the two shapes are (M: node masses)
- **σ** → shape change per unit of z, the subspace's size
- **≈** → soft: a penalty, not a hard rule (Prop. B.1)

> **L(θ) = 𝔼 z, z′ [ E(f(z)) ]  +  λ 𝔼 z, z′ [ log( |f(z) − f(z′)|_M ⁄ σ|z − z′| )² ]**

*Read it as:* make shapes cheap (left) while keeping them spread out (right); λ sets the balance.

- **𝔼 z, z′** → average over random pairs of inputs
- **E(f(z))** → energy of one shape
- **|f(z) − f(z′)|_M** → distance between the two shapes
- **σ|z − z′|** → the distance we asked for
- **log( )²** → 0 when they match; half or double cost the same

> **f(z) = ρ·MLP(z) + (1 − ρ)·q_seed**

*Read it as:* blend the network with one known shape, and slide the blend from all seed to all network.

- **ρ = 0** → every output is the seed (straight bar, E = 9), never nonsense
- **ρ = 1** → the plain network
- **ρ rises** → linearly over the first half of training
- **ρσ** → the target spread grows with it

### Geometry primer: distances between every pair

Think of each shape as one point in a 36-dimensional space. The **distance** between two shapes is the length of their difference, `(q_i - q_j).norm()`, exactly as for 2D points, just with 36 numbers.

The distance term compares, for every pair in the batch, two distances:

- in latent space: |z_i − z_j| (how far apart we asked them to be, before scaling by σ),
- in shape space: |q_i − q_j| (how far apart they came out).

`torch.triu_indices(B, B, 1)` lists every pair once: for B = 4 it gives i = [0, 0, 0, 1, 1, 2] and j = [1, 2, 3, 2, 3, 3]. Indexing `q[i]` and `q[j]` lines the pairs up row by row, so one subtraction handles all of them.

The cost per pair is (log of the ratio)²: 0 when the ratio is 1, and the same for "twice too far" as for "half as far".

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

Distances for every pair in shape space, distances for every pair in latent space, then the mean of (log dq − log(σ dz))². For `subspace_map`, blend the network output with the seed using ρ.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`q[i] - q[j]` has shape `(pairs, 36)`; `.norm(dim=1)` gives `(pairs,)`. Same for `z`. `torch.log` works element by element. In `subspace_map`, `net(z)` has shape `(batch, 36)` and `q0` has shape `(36,)`: broadcasting adds the seed to every row.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
dq = (q[i] - q[j]).norm(dim=1)
dz = (z[i] - z[j]).norm(dim=1)
return ((torch.log(dq) - torch.log(sigma * dz)) ** 2).mean()
# and in subspace_map:
return rho * net(z) + (1 - rho) * q0
```

</details>

In [ ]:
def isometry_loss(q, z, sigma):
    # mean over all pairs (i < j) of ( log |q_i - q_j| - log(sigma |z_i - z_j|) )^2
    i, j = torch.triu_indices(len(z), len(z), 1)
    # YOUR CODE HERE
    raise NotImplementedError("Fill in this step")

def subspace_map(net, z, rho, q0):
    # YOUR CODE HERE
    raise NotImplementedError("Fill in this step")

def train_subspace(d=1, sigma=3.0, lam=1.0, steps=3000, batch=32,
                   energy_fn=energy, q0=q_seed, ramp=0.5, verbose=True):
    net = SubspaceNet(d, out=q0.numel())
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    t0 = time.time()
    for it in range(steps):
        rho = min(1.0, (it + 1) / (ramp * steps))       # seeded schedule: 0 -> 1 over the first half
        z = torch.randn(batch, d)
        q = subspace_map(net, z, rho, q0)
        loss = energy_fn(q).mean() + lam * isometry_loss(q, z, rho * sigma)
        opt.zero_grad(); loss.backward(); opt.step()
        if verbose and it % 1000 == 0:
            print(f"step {it:5d} | rho {rho:.2f} | loss {loss.item():.3f}")
    if verbose: print(f"trained in {time.time() - t0:.1f} s")
    return net

torch.manual_seed(0)
net1 = train_subspace(d=1, sigma=3.0)

**Check your answer:** run this cell. Every line should say PASS.

In [ ]:
zz = torch.tensor([[0.0], [1.0], [3.0]])
qq = torch.cat([zz, torch.zeros(3, 1)], 1)                  # shapes exactly 1x as far apart as the z's
check(abs(isometry_loss(qq, zz, 1.0).item()) < 1e-6, "perfect distance keeping costs 0")
check(abs(isometry_loss(2 * qq, zz, 1.0).item() - math.log(2) ** 2) < 1e-5, "twice too far costs (log 2)^2 = 0.48")
check(abs(isometry_loss(qq / 2, zz, 1.0).item() - math.log(2) ** 2) < 1e-5, "half as far costs the same 0.48")
lin = nn.Linear(1, n)
check(torch.allclose(subspace_map(lin, zz, 0.0, q_seed), q_seed.expand(3, n)), "rho = 0 gives the seed for every z")

In [ ]:
zs = torch.linspace(-2.5, 2.5, 11)[:, None]
with torch.no_grad():
    Q = net1(zs); E = energy(Q); mid = full_nodes(Q)[:, N // 2, 1]

fig, axs = plt.subplots(1, 2, figsize=(15, 3.2))
for k, q in enumerate(Q): draw(axs[0], q, color=cm(k / (len(Q) - 1)))
axs[0].set_title("neural subspace, z from -2.5 (purple) to 2.5 (yellow)")
axs[1].plot(zs[:, 0], E, "o-"); axs[1].set_xlabel("z"); axs[1].set_ylabel("energy"); axs[1].set_title("energy along the subspace")
plt.show()
for z, e, m in zip(zs[:, 0], E, mid):
    print(f"z = {z:+.1f} | energy {e:6.3f} | middle node height {m:+.3f}")

**Question:** where in z are the two buckled states? How high is the energy bump between them, and what shape does the bar take there? Compare that bump with the straight seed (about 9).

## Task 5 · What σ does  *(code break 5)*

> **Slides:** Part 3, "Choosing σ and λ". The question slide before this break asks what you expect for very small and very large σ.

σ sets how big the subspace is; λ sets how strictly distances are kept. Train three subspaces with σ = 0.3, 1 and 3, and measure the paper's health check (Appendix C.1): the mean |log ratio| from Eq. 2 on fresh samples. Near 0 means distances are kept; far from 0 means λ is too weak for that σ.

**Your job:** compute `ratio`, the mean of |log(|f(z) − f(z′)| / (σ|z − z′|))| over all pairs `i, j`.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

This is the inside of `isometry_loss` again, but with the absolute value instead of the square: the mean of |log dq − log(σ dz)|.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`i, j` are already given. Build `dq` and `dz` exactly as in Task 4, then `.abs().mean()`.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
ratio = (torch.log((q[i] - q[j]).norm(dim=1)) - torch.log(sigma * (z[i] - z[j]).norm(dim=1))).abs().mean()
```

</details>

In [ ]:
results = {}
for sigma in [0.3, 1.0, 3.0]:
    torch.manual_seed(0)
    net = train_subspace(d=1, sigma=sigma, steps=2000, verbose=False)
    with torch.no_grad():
        z = torch.randn(256, 1); q = net(z)
        i, j = torch.triu_indices(256, 256, 1)
        # YOUR CODE HERE
        raise NotImplementedError("Fill in this step")
        mid = full_nodes(net(torch.linspace(-2.5, 2.5, 11)[:, None]))[:, N // 2, 1]
        results[sigma] = (energy(q).mean().item(), ratio.item(), mid.min().item(), mid.max().item())
    print(f"sigma {sigma:3.1f} | mean energy {results[sigma][0]:6.3f} | mean |log ratio| {results[sigma][1]:.3f} "
          f"| middle node from {results[sigma][2]:+.2f} to {results[sigma][3]:+.2f}")

**Check your answer:** run this cell. Every line should say PASS.

In [ ]:
check(results[0.3][1] < results[3.0][1], "the ratio is worse at sigma = 3 than at sigma = 0.3")
check(0.05 < results[0.3][1] < 0.2, f"sigma = 0.3 ratio is about 0.10 (yours: {results[0.3][1]:.2f})")

**Question:** which σ reaches both buckled states, and what does it cost? What does the health check say at σ = 3?

## Task 6 · Linear vs. neural, far from rest  *(code break 6)*

> **Slides:** Part 3, "Back to Modal Analysis" and "Linear vs. Neural on Our Bar". The question slide before this break asks which path hits the higher energy peak.

The paper proves that Eq. 4 becomes linear modal analysis when λ → ∞ and σ → 0 (Proposition 3.1). Now compare the two subspaces at full size: the linear map from Task 2 and `net1` from Task 4, both over z ∈ [−2.5, 2.5]. Also try the most obvious linear path of all: the straight line from buckled-down to buckled-up.

**Your job:** build the straight-line shapes `(1 − t) q_down + t q_up` for the 51 values of `t`.

### Geometry primer: the straight line between two shapes

The straight path from shape A to shape B is (1 − t) A + t B, with t from 0 to 1: at t = 0 you are at A, at t = 1 at B, and halfway at t = 0.5 you are at the average. Every node moves in a straight line, at constant speed.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

Blend `q_down` and `q_up` with weights (1 − t) and t, for all 51 values of t at once.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`t` has shape `(51, 1)`; `q_down` has shape `(36,)`. Multiplying them broadcasts to `(51, 36)`: one shape per row.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
Q_line = (1 - t) * q_down + t * q_up
```

</details>

In [ ]:
zs = torch.linspace(-2.5, 2.5, 51)[:, None]
t = torch.linspace(0, 1, 51)[:, None]
with torch.no_grad():
    E_neural = energy(net1(zs))
    E_linear = energy(linear_map(zs))
    # YOUR CODE HERE
    raise NotImplementedError("Fill in this step")
    E_straight = energy(Q_line)

fig, axs = plt.subplots(1, 2, figsize=(15, 3.2))
axs[0].plot(zs, E_neural, label="neural subspace"); axs[0].plot(zs, E_linear, label="linear modal subspace")
axs[0].set_yscale("log"); axs[0].set_xlabel("z"); axs[0].set_ylabel("energy (log)"); axs[0].legend()
axs[1].plot(t, E_straight); axs[1].set_xlabel("fraction of the way from down to up"); axs[1].set_title("straight line between the two stable shapes")
plt.show()
print(f"max energy, neural subspace : {E_neural.max().item():8.2f}")
print(f"max energy, linear subspace : {E_linear.max().item():8.2f}")
print(f"max energy, straight line   : {E_straight.max().item():8.2f}")

**Check your answer:** run this cell. Every line should say PASS.

In [ ]:
check(Q_line.shape == (51, n), "51 shapes of 36 numbers")
check(torch.allclose(Q_line[0], q_down) and torch.allclose(Q_line[-1], q_up), "the line starts at buckled-down and ends at buckled-up")

**Question:** why does the linear subspace's energy climb so fast, and why can no straight line reach buckled-up cheaply?

## Task 7 · Simulate inside the subspace  *(code break 7)*

> **Slides:** Part 4, "Simulating Inside the Subspace". The question slide before this break asks which push snaps the bar through.

Implicit Euler in the subspace (Eq. 6): each time step finds the latent ẑ that minimises

m |f(z) − q̃|² / (2h²) + E(f(z)) − F · f(z),

where q̃ = 2q_t − q_(t−1) is where the shape would go if it kept its velocity, and F is an external force. We search over **one number**, z, with L-BFGS, instead of 36.

We start the bar buckled up and push its middle down for 15 steps, with a push of 1.5 and of 3.

**Your job:** write `obj` inside `closure`.

### Reading the equation

**Implicit Euler in z (Eq. 6)**

> **ẑ = argmin_z |f(z) − q̃|²_M ⁄ 2h² + E(f(z))**

*Read it as:* pick the z whose shape best balances 'keep moving as before' against 'keep the bar relaxed'.

- **argmin_z** → try values of z, keep the best
- **q̃ = 2q_t − q_(t−1)** → where momentum alone would carry the shape
- **|f(z) − q̃|² ⁄ 2h²** → cost of leaving that path (h: time step)
- **E(f(z))** → the bar's own energy
- **q_(t+1) = f(ẑ)** → the new shape; only d numbers searched

### Physics primer: one implicit Euler step as a tug of war

Each time step picks the next shape by balancing three pulls:

1. **Inertia:** keep moving as you were. q̃ = 2q_t − q_(t−1) is where the shape would go at constant velocity; the term m |q − q̃|² / (2h²) charges for leaving that path.
2. **The bar's own energy** E(q): bending and stretching push back.
3. **The push:** an external force F does work F · q. Subtracting it makes moving along F cheaper.

Minimising the sum gives the next shape. In the subspace we only search over z.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

Add up three numbers: the inertia term, the energy, and minus the work done by F.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`q`, `q_tilde` and `F` are all shaped like one flattened shape. `((q - q_tilde) ** 2).sum()` is |q − q̃|². `energy(q)` returns a one-element batch, so add `.sum()` to get a plain number. `(F * q).sum()` is the dot product F · q.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
obj = m * ((q - q_tilde) ** 2).sum() / (2 * h * h) + energy(q).sum() - (F * q).sum()
```

</details>

In [ ]:
zs = torch.linspace(-3, 3, 301)[:, None]
with torch.no_grad():
    Qs = net1(zs); Es = energy(Qs); mids = full_nodes(Qs)[:, N // 2, 1]
z_up = zs[mids > 0][Es[mids > 0].argmin()].view(1, 1)      # latent of the buckled-up shape

def simulate(push, steps=80, h=0.02, m=1.0 / N):
    z, z_prev, heights = z_up.clone(), z_up.clone(), []
    for t in range(steps):
        with torch.no_grad():
            q_tilde = 2 * net1(z) - net1(z_prev)        # inertial guess
        F = torch.zeros(N - 2, 2)
        if t < 15: F[N // 2 - 1:N // 2 + 1, 1] = -push  # push the middle nodes down
        F = F.reshape(-1)
        z_new = z.clone().requires_grad_(True)
        opt = torch.optim.LBFGS([z_new], max_iter=50, line_search_fn="strong_wolfe")
        def closure():
            opt.zero_grad()
            q = net1(z_new)
            # YOUR CODE HERE
            raise NotImplementedError("Fill in this step")
            obj.backward(); return obj
        opt.step(closure)
        z_prev, z = z, z_new.detach()
        with torch.no_grad(): heights.append(full_nodes(net1(z))[0, N // 2, 1].item())
    return heights

for push in [1.5, 3.0]:
    hts = simulate(push)
    plt.plot(hts, label=f"push {push}")
    print(f"push {push}: final middle-node height {hts[-1]:+.2f}")
plt.axhline(0, color="gray", lw=0.5); plt.xlabel("time step"); plt.ylabel("middle node height"); plt.legend(); plt.show()

**Question:** what happens with each push, and why is each time step cheap?

## Task 8 · One network for every pin distance  *(code break 8)*

> **Slides:** Part 4, "Conditional Subspaces". The question slide before this break asks how one network could handle every pin distance.

Eq. 7 feeds an extra condition c into the network: q = f([z, c]). Here c is the distance between the pins, from 0.6 to 0.9. During training every sample gets a random c, so one network learns the subspace for every squeeze.

**Your job:** build the network input `inp` by joining z and c along the feature dimension.

### Reading the equation

> **q = f_θ([z, c]),   c ∈ ℝᵐ**

*Read it as:* one network, one subspace for every setting c, fed in next to z and drawn at random while training.

- **z** → where you are inside the subspace
- **c** → a setting of the system: here the pin distance 0.6 to 0.9
- **[z, c]** → the two joined into one input
- **paper** → conditions on stiffness and pin positions (Fig. 5)

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

The network should see two numbers per sample: its latent z and its pin distance c. Put them side by side.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`z` has shape `(32, 1)` but `c` has shape `(32,)`. Give `c` a second axis with `c[:, None]`, then `torch.cat([...], dim=1)` to get `(32, 2)`.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
inp = torch.cat([z, c[:, None]], 1)
```

</details>

In [ ]:
def train_conditional(steps=4000, sigma=3.0, lam=1.0, batch=32):
    net = SubspaceNet(1, cond_dim=1)
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    for it in range(steps):
        rho = min(1.0, (it + 1) / (0.5 * steps))
        z = torch.randn(batch, 1)
        c = 0.6 + 0.3 * torch.rand(batch)
        xs = torch.linspace(0, 1, N)[1:-1][None] * c[:, None] * L
        q0 = torch.stack([xs, torch.zeros_like(xs)], -1).reshape(batch, -1)   # straight seed for each c
        # YOUR CODE HERE
        raise NotImplementedError("Fill in this step")
        q = rho * net(inp) + (1 - rho) * q0
        loss = energy(q, c).mean() + lam * isometry_loss(q, z, rho * sigma)
        opt.zero_grad(); loss.backward(); opt.step()
    return net

torch.manual_seed(0)
net_c = train_conditional()
fig, axs = plt.subplots(1, 3, figsize=(16, 3))
for ax, cc in zip(axs, [0.6, 0.75, 0.9]):
    with torch.no_grad():
        zz = torch.linspace(-2.5, 2.5, 7)[:, None]
        Q = net_c(torch.cat([zz, torch.full_like(zz, cc)], 1))
    for k, q in enumerate(Q): draw(ax, q, c=cc, color=cm(k / 6))
    ax.set_title(f"pins {cc} apart")
plt.show()

## Task 9 · A floor, as a penalty  *(code break 9)*

> **Slides:** Part 4, "Constraints and Collisions as Penalties"

Eq. 9 turns constraints into energy: an inequality C(q) ≥ 0 costs w · min(C(q), 0)². Put a floor at y = −0.2 and add that penalty for every node. Nothing else in the method changes.

**Your job:** write `penalty`: w times the sum over nodes of min(y − floor, 0)².

### Reading the equation

> **w |C(q)|² + w′ |min(C′(q), 0)|²**

*Read it as:* turn every rule into extra energy, charged only when the rule is broken.

- **C(q) = 0** → a rule that must hold exactly: joints stay joined
- **C′(q) ≥ 0** → a one-sided rule: stay above the floor
- **min( , 0)** → only violations cost anything
- **w, w′** → how strict; finite, so training can pass through
- **our bar** → floor at y = −0.2: lowest points stop at −0.20

### Geometry primer: a one-sided penalty

For each node, y − floor is positive above the floor and negative below it. We only want to charge the negative part:

- `torch.clamp(x, max=0)` keeps negative values and replaces positive ones by 0,
- squaring makes the cost grow smoothly the deeper a node goes,
- multiplying by a big weight w makes it strong.

Above the floor the penalty is exactly 0, so shapes that respect the floor are not affected at all.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

For every node: how far below the floor is it (0 if above)? Square that, add over the nodes, multiply by w.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`X[..., 1]` is the height of every node, shape `(batch, 20)`. After clamping and squaring, `.sum(1)` adds over the nodes and leaves one number per shape.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
penalty = w * (torch.clamp(X[..., 1] - floor, max=0) ** 2).sum(1)
```

</details>

In [ ]:
def floor_energy(q, floor=-0.2, w=1000.0):
    X = full_nodes(q)
    # YOUR CODE HERE
    raise NotImplementedError("Fill in this step")
    return energy(q) + penalty

torch.manual_seed(0)
net_floor = train_subspace(d=1, sigma=3.0, energy_fn=floor_energy, verbose=False)
with torch.no_grad():
    Q = net_floor(torch.linspace(-2.5, 2.5, 11)[:, None])
fig, ax = plt.subplots(figsize=(6, 3))
for k, q in enumerate(Q): draw(ax, q, color=cm(k / 10))
ax.axhline(-0.2, color="k", lw=2); ax.set_title("subspace with a floor at y = -0.2"); plt.show()
print("lowest point of each shape:", [round(v, 2) for v in full_nodes(Q)[..., 1].min(1).values.tolist()])

**Check your answer:** run this cell. Every line should say PASS.

In [ ]:
check(abs((floor_energy(q_up[None]) - energy(q_up[None])).item()) < 1e-9, "a shape above the floor pays no penalty")
q_low = q_up.clone().view(-1, 2); q_low[8, 1] = -0.3        # buckled-up shape, one node pushed 0.1 below the floor
extra = (floor_energy(q_low.reshape(1, -1)) - energy(q_low.reshape(1, -1))).item()
check(abs(extra - 10.0) < 1e-3, f"one node 0.1 below the floor costs 1000 x 0.1^2 = 10 (yours: {extra:.2f})")

## Task 10 · Rigid bodies as penalties  *(code break 10)*

> **Slides:** Part 4, "Rigid Bodies Without Special Code"

The paper handles rigid bodies with no special code. Each body is a free matrix [R | t]: in 2D, a 2 × 2 block R and a translation t, so 6 numbers per body. A penalty |RᵀR − I|² pushes R toward a rotation, and joints are penalties that pull matching end points together. Everything else (network, distance term, seed) is unchanged.

First, one stiff link of length 1, pinned at its top end and hanging under gravity. Then a chain of 4 links.

**Your job:** write the rigidity penalty `E_rigid`: w_rig times the sum over bodies of |RᵀR − I|² (sum of squared entries).

### Reading the equation

> **[R | t] ∈ ℝ³ˣ⁴,  penalty |RᵀR − I|²**

*Read it as:* store each body as a free matrix, and charge it for anything that is not a pure rotation.

- **t** → where the body is
- **R** → how it is turned (3 × 3)
- **RᵀR = I** → true exactly for rotations; stretch and shear break it
- **collisions** → penalties on signed distances
- **chain** → linear modes show no useful motion even at d = 8

### Geometry primer: when is a 2 × 2 matrix a rotation?

A rigid link can only rotate and move; it cannot stretch or shear. In 2D its orientation is a matrix R whose two columns are where the x and y axes end up.

R is a pure rotation exactly when both columns have length 1 and are perpendicular. That is the same as **RᵀR = I**, because the entries of RᵀR are the dot products of the columns with each other:

| | RᵀR |
|---|---|
| rotation | [[1, 0], [0, 1]] |
| stretch x by 2 | [[4, 0], [0, 1]]: a diagonal entry is not 1 |
| shear | [[1, 0.5], [0.5, 1.25]]: off-diagonal entries are not 0 |

So |RᵀR − I|² (the sum of its squared entries) is 0 for a rotation and positive for anything else: a perfect rigidity penalty.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

For every link: compute RᵀR, subtract the identity, square every entry, and add everything up. Multiply by w_rig.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`R` has shape `(batch, K, 2, 2)`. `R.transpose(-1, -2)` is Rᵀ for every link at once, and `@` multiplies the matching 2 × 2 matrices. Sum over the last three axes `(-1, -2, -3)`: both matrix axes and the links, leaving one number per shape.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
E_rigid = w_rig * ((R.transpose(-1, -2) @ R - I) ** 2).sum((-1, -2, -3))
```

</details>

In [ ]:
w_rig, w_joint, g_rb = 200.0, 2000.0, 1.0

def make_rigid_system(K, ell):
    def unpack(q):
        P = q.view(q.shape[0], K, 2, 3)
        return P[..., :2], P[..., 2]                         # R: (B, K, 2, 2), t: (B, K, 2)
    def ends(q):
        R, t = unpack(q)
        top, bot = torch.tensor([-ell / 2, 0.0]), torch.tensor([ell / 2, 0.0])
        return t + R @ top, t + R @ bot                      # top and bottom end of every link
    def rb_energy(q):
        R, t = unpack(q); top, bot = ends(q)
        I = torch.eye(2)
        # YOUR CODE HERE
        raise NotImplementedError("Fill in this step")
        E_pin = w_joint * (top[:, 0] ** 2).sum(-1)                   # first link pinned at the origin
        E_joint = w_joint * ((bot[:, :-1] - top[:, 1:]) ** 2).sum((-1, -2))
        E_grav = g_rb * t[..., 1].sum(-1)
        return E_rigid + E_pin + E_joint + E_grav
    R_down = torch.tensor([[0.0, 1.0], [-1.0, 0.0]])        # link axis pointing straight down
    q_rest = torch.stack([torch.cat([R_down, torch.tensor([[0.0], [-ell * (k + 0.5)]])], 1)
                          for k in range(K)]).reshape(-1)
    return unpack, ends, rb_energy, q_rest

def rigidity_error(unpack, q):
    R, _ = unpack(q)
    return ((R.transpose(-1, -2) @ R - torch.eye(2)) ** 2).sum((-1, -2)).sqrt().mean(-1)

# one stiff link, pinned at the top
unpack1, ends1, link_energy, link_rest = make_rigid_system(K=1, ell=1.0)
torch.manual_seed(0)
net_link = train_subspace(d=1, sigma=1.0, energy_fn=link_energy, q0=link_rest, verbose=False)

zs = torch.linspace(-2.5, 2.5, 11)[:, None]
with torch.no_grad():
    Q = net_link(zs); top, bot = ends1(Q)
fig, ax = plt.subplots(figsize=(5, 3))
for k in range(len(Q)):
    ax.plot([top[k, 0, 0], bot[k, 0, 0]], [top[k, 0, 1], bot[k, 0, 1]], "-o", color=cm(k / 10))
ax.set_aspect("equal"); ax.set_title("neural subspace of a pinned stiff link"); plt.show()
for z, b, r in zip(zs[:, 0], bot[:, 0], rigidity_error(unpack1, Q)):
    print(f"z = {z:+.1f} | swing angle {math.degrees(math.atan2(b[0], -b[1])):+4.0f} deg | rigidity error {r:.3f}")

**Check your answer:** run this cell. Every line should say PASS.

In [ ]:
c, s = math.cos(0.7), math.sin(0.7)
q_rot = torch.tensor([[c, -s, 0.0], [s, c, 0.0]]).reshape(1, -1)             # one link, rotated by 0.7 rad
q_str = torch.tensor([[2.0, 0.0, 0.0], [0.0, 1.0, 0.0]]).reshape(1, -1)      # one link, stretched 2x along x
R_rot, R_str = q_rot.view(1, 1, 2, 3)[..., :2], q_str.view(1, 1, 2, 3)[..., :2]
pen = lambda R: ((R.transpose(-1, -2) @ R - torch.eye(2)) ** 2).sum().item()   # the same formula you wrote
check(pen(R_rot) < 1e-10, "a rotation matrix has R^T R = I, penalty 0")
check(abs(pen(R_str) - 9.0) < 1e-6, "stretching x by 2 gives (4 - 1)^2 = 9")

Joints are penalties too. The same code with 4 links gives a chain; here with d = 2.

In [ ]:
unpack4, ends4, chain_energy, chain_rest = make_rigid_system(K=4, ell=0.25)
torch.manual_seed(0)
net_chain = train_subspace(d=2, sigma=1.5, energy_fn=chain_energy, q0=chain_rest, verbose=False)
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
sweep = torch.linspace(-2.5, 2.5, 7)
with torch.no_grad():
    for a, ax in enumerate(axs):
        Z = torch.zeros(7, 2); Z[:, a] = sweep
        Q = net_chain(Z); top, bot = ends4(Q)
        for k in range(7):
            P = torch.cat([top[k], bot[k, -1:]], 0)
            ax.plot(P[:, 0], P[:, 1], "-o", ms=4, color=cm(k / 6))
        ax.set_aspect("equal"); ax.set_xlim(-0.8, 0.8); ax.set_ylim(-1.1, 0.1)
        ax.set_title(f"sweep z{a + 1}: max rigidity error {rigidity_error(unpack4, Q).max().item():.3f}")
plt.suptitle("a 4-link chain, d = 2: links stay rigid, joints stay joined"); plt.show()

**Question:** does the link stay rigid as it swings? What did you have to change in the method to handle rigid bodies?

## Task 11 · Rotation: linear vs. neural  *(code break 11)*

> **Slides:** Part 5, "Neural Subspaces Can Rotate; Linear Ones Cannot". The question slide before this break asks why a linear subspace might struggle with a swinging object.

The paper's Fig. 7 pins a stiff object at one point and compares subspaces. Do the same with our link: build the linear modal subspace at rest (softest Hessian eigenvector, σ = 1) and compare it with `net_link` from Task 10.

**Your job:** compute the Hessian of `link_energy` at `link_rest`, take its softest eigenvector `A_link`, and define the linear map.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

Exactly Task 2 again, for the link: Hessian at rest, eigen-decomposition, keep the softest eigenvector.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

Wrap the energy so it takes one 6-number shape: `lambda q: link_energy(q[None]).sum()`. Keep `A_link` with shape `(6, 1)` using `[:, :1]`.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
H_link = torch.autograd.functional.hessian(lambda q: link_energy(q[None]).sum(), link_rest)
ev_link, V_link = torch.linalg.eigh(H_link)
A_link = V_link[:, :1]
```

</details>

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Fill in this step")
def linear_link(z, sigma=1.0):
    return link_rest + sigma * z @ A_link.T

zs = torch.linspace(-2.5, 2.5, 11)[:, None]
with torch.no_grad():
    for name, Q in [("neural", net_link(zs)), ("linear", linear_link(zs))]:
        top, bot = ends1(Q)
        length = (bot - top)[:, 0].norm(dim=-1)
        ang = torch.rad2deg(torch.atan2(bot[:, 0, 0], -bot[:, 0, 1]))
        print(f"{name:6s} | swing {ang.min().item():+4.0f} to {ang.max().item():+4.0f} deg "
              f"| link length {length.min().item():.2f} to {length.max().item():.2f} "
              f"| max energy {link_energy(Q).max().item():8.2f}")

fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
with torch.no_grad():
    for ax, (name, Q) in zip(axs, [("neural", net_link(zs)), ("linear", linear_link(zs))]):
        top, bot = ends1(Q)
        for k in range(len(Q)):
            ax.plot([top[k, 0, 0], bot[k, 0, 0]], [top[k, 0, 1], bot[k, 0, 1]], "-o", color=cm(k / 10))
        ax.set_aspect("equal"); ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.1, 0.3); ax.set_title(f"{name} subspace")
plt.show()

**Check your answer:** run this cell. Every line should say PASS.

In [ ]:
check(H_link.shape == (6, 6), "the link has 6 numbers, so H_link is 6 x 6")
check(A_link.shape == (6, 1), "A_link is one column")

**Question:** what happens to the link's length along the linear subspace, and why?

## Task 12 · Sampled shapes as training data  *(code break 12)*

> **Slides:** Part 5, "Data-Free vs. a Trained Autoencoder"

Data-driven methods need recorded shapes. A data-free subspace can *make* them: the paper samples its subspace to train a downstream model (AutoDef, Fig. 9). Here we sample 4,000 shapes from `net1`, then fit two classic data-driven models with one latent number each:

- **PCA**: a linear subspace (the mean plus the top principal direction).
- **An autoencoder**: an encoder MLP to one number and a decoder MLP back.

**Your job:** project the data onto the top principal direction and back (PCA with d = 1).

### Geometry primer: projecting onto one direction

PCA with one direction replaces every shape by its "shadow" on a single straight line through the average shape μ, along a unit direction v:

1. move the origin to the average: x − μ,
2. measure how far along v you are: s = (x − μ) · v (one number, a dot product),
3. go back to shape space: μ + s v.

The line is straight, so if the real shapes lie on a curve, every shadow misses the curve a bit, and the shadows can be shapes the bar would never take.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

For every shape: subtract the mean, take the dot product with v1, then rebuild mean + (that number) × v1.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`data - mu` has shape `(4000, 36)`; `@ v1` gives `(4000,)`, one coordinate per shape. To multiply each coordinate by the whole direction, reshape: `[:, None]` gives `(4000, 1)` and `v1[None]` gives `(1, 36)`; the product is `(4000, 36)`.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
pca_rec = mu + ((data - mu) @ v1)[:, None] * v1[None]
```

</details>

In [ ]:
torch.manual_seed(0)
with torch.no_grad():
    data = net1(torch.randn(4000, 1))                    # the "dataset", made without any simulation

mu = data.mean(0)
U, S, Vt = torch.linalg.svd(data - mu, full_matrices=False)
v1 = Vt[0]                                              # top principal direction, shape (n,)
# YOUR CODE HERE
raise NotImplementedError("Fill in this step")

enc = nn.Sequential(nn.Linear(n, 64), nn.ELU(), nn.Linear(64, 64), nn.ELU(), nn.Linear(64, 1))
dec = SubspaceNet(1)
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), 1e-3)
for it in range(3000):
    batch = data[torch.randint(0, len(data), (128,))]
    loss = ((dec(enc(batch)) - batch) ** 2).sum(1).mean()
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    ae_rec = dec(enc(data))

for name, rec in [("PCA, d = 1", pca_rec), ("autoencoder, d = 1", ae_rec)]:
    err = (rec - data).norm(dim=1).mean().item()
    print(f"{name:20s} | mean reconstruction error {err:.3f} | median energy {energy(rec).median().item():7.2f} "
          f"| max energy {energy(rec).max().item():8.1f}")
print(f"{'the data itself':20s} | median energy {energy(data).median().item():7.2f}")

**Check your answer:** run this cell. Every line should say PASS.

In [ ]:
check(pca_rec.shape == data.shape, "one reconstruction per shape")
test = mu + 3 * v1
check(torch.allclose(mu + ((test - mu) @ v1) * v1, test, atol=1e-5), "a point already on the line is reconstructed exactly")
check(abs(((pca_rec - mu) @ v1 - (data - mu) @ v1).abs().max().item()) < 1e-4, "projecting keeps the coordinate along v1")

**Question:** which model reproduces the shapes, and what did each one need?

## Task 13 · Two latent numbers  *(code break 13)*

> **Slides:** Part 5, "Going Deeper: More Latent Dimensions"

Train a d = 2 subspace and draw a 5 × 5 grid of shapes, with the energy of each.

**Your job:** build `grid`, the 25 latent points (z₁, z₂) with both values from `vals`, as a (25, 2) tensor. `torch.cartesian_prod` does this.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

You need every combination (z1, z2) with both values taken from `vals`: 5 × 5 = 25 points.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`torch.cartesian_prod(vals, vals)` returns exactly that as a `(25, 2)` tensor, z1 changing slowest. That ordering puts z1 along the rows of the plot.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
grid = torch.cartesian_prod(vals, vals)
```

</details>

In [ ]:
torch.manual_seed(0)
net2 = train_subspace(d=2, sigma=3.0, steps=3000, verbose=False)
vals = torch.linspace(-2, 2, 5)
# YOUR CODE HERE
raise NotImplementedError("Fill in this step")
with torch.no_grad():
    Q = net2(grid); E = energy(Q)
fig, axs = plt.subplots(5, 5, figsize=(14, 7))
for k, ax in enumerate(axs.flat):
    draw(ax, Q[k]); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"E={E[k].item():.2f}", fontsize=8)
plt.suptitle("d = 2 subspace: rows z1, columns z2"); plt.tight_layout(); plt.show()

**Check your answer:** run this cell. Every line should say PASS.

In [ ]:
check(grid.shape == (25, 2), "25 latent points with 2 numbers each")
check(grid[0].tolist() == [-2.0, -2.0] and grid[1].tolist() == [-2.0, -1.0], "z1 changes slowest, z2 fastest")

**Question:** what does each latent direction seem to control? Which corner is most expensive?

## Task 14 · Seeds and restarts  *(code break 14)*

> **Slides:** Part 5, "Limitations". The question slide before this break asks what could go wrong without a dataset to check against.

The paper says any reasonable seed works, but training is non-convex. Train from the straight seed and from the buckled-**up** shape, each with three random initialisations, and check which runs cover both states.

**Your job:** set `covers` to True when the middle node goes below −0.28 and above +0.28 (the two buckled states sit near ±0.32) somewhere along the subspace.

**Stuck?** Open the hints one at a time.

<details><summary><b>Hint 1 · the idea</b></summary>

A run covers both states if the middle node goes clearly below 0 somewhere and clearly above 0 somewhere along the subspace.

</details>

<details><summary><b>Hint 2 · shapes and functions</b></summary>

`mid` holds the middle-node height of 51 shapes along z. `mid.min()` and `mid.max()` are tensors; compare them with −0.28 and +0.28 and combine with `and`, wrapped in `bool(...)`.

</details>

<details><summary><b>Hint 3 · almost the answer</b></summary>

```python
covers = bool(mid.min() < -0.28 and mid.max() > 0.28)
```

</details>

In [ ]:
for name, q0 in [("straight seed", q_seed), ("buckled-up seed", q_up)]:
    for s in [0, 1, 2]:
        torch.manual_seed(s)
        net = train_subspace(d=1, sigma=3.0, steps=3000, q0=q0, verbose=False)
        with torch.no_grad():
            Q = net(torch.linspace(-2.5, 2.5, 51)[:, None]); mid = full_nodes(Q)[:, N // 2, 1]; E = energy(Q)
        # YOUR CODE HERE
        raise NotImplementedError("Fill in this step")
        print(f"{name:16s} | init {s} | middle node {mid.min().item():+.2f} to {mid.max().item():+.2f} "
              f"| max energy {E.max().item():5.2f} | covers both states: {covers}")

**Question:** do all runs cover both buckled states? How would you notice a bad run in a system you cannot draw?

## Discussion questions

1. The method needs only an energy and a seed. Which systems have no clean energy function?
2. Why use a *logarithm* in the distance term, and why compare only pairs inside a batch?
3. When would you still choose linear modes, or an autoencoder trained on data?
4. Each subspace fits one system. What would one model for many systems need?
5. Session 1 link: these energies ignore rotating or shifting the whole object. How could an equivariant subspace map use that?